# Modeling Experiments — Prediction Market Forecasting

End-to-end pipeline: data ingestion, feature engineering, model training, evaluation, and visualization.

**Pipeline:**
1. Fetch top Polymarket contracts by volume
2. Build endogenous + exogenous feature matrices
3. Train baselines, tree ensembles, and (optionally) deep learning models
4. Evaluate: metrics, statistical tests, trading simulation
5. Visualize: figures and tables for the paper

In [ ]:
import sys, json, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="pytorch_lightning")

sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(name)s %(levelname)s %(message)s")

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 120})

# Configuration
N_CONTRACTS = 20       # number of top contracts to fetch
MIN_VOLUME = 1_000_000 # USD volume threshold
N_PAGES = 20           # Gamma API pages to scan

print("Setup OK")

## 1. Data Ingestion — Top Polymarket Contracts

In [ ]:
import requests as req

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"

# 1a. Discover active markets, sorted by volume (sync — reliable in all contexts)
all_markets = []
for page in range(N_PAGES):
    resp = req.get(f"{GAMMA_BASE}/markets",
                   params={"limit": 100, "offset": page * 100, "active": "true", "closed": "false"},
                   timeout=30)
    batch = resp.json()
    if not batch:
        break
    all_markets.extend(batch)

markets_df = pd.DataFrame(all_markets)
markets_df["volumeNum"] = pd.to_numeric(markets_df["volumeNum"], errors="coerce").fillna(0)
markets_df = markets_df.sort_values("volumeNum", ascending=False).reset_index(drop=True)

top_markets = markets_df[markets_df["volumeNum"] >= MIN_VOLUME].head(N_CONTRACTS)
print(f"Scanned {len(markets_df)} markets, selected top {len(top_markets)} with volume >= ${MIN_VOLUME:,.0f}")
top_markets[["question", "volumeNum"]].head(10)

In [ ]:
# 1b. Fetch hourly price history for each contract (Yes side)

histories = {}
for _, row in top_markets.iterrows():
    token_ids = json.loads(row.get("clobTokenIds", "[]")) if isinstance(row.get("clobTokenIds"), str) else (row.get("clobTokenIds") or [])
    question = str(row.get("question", ""))[:60]
    cid = row.get("conditionId", question)
    end_date = row.get("endDate") or row.get("endDateIso")
    volume = row.get("volumeNum", 0)
    
    if not token_ids:
        continue
    
    try:
        resp = req.get(f"{CLOB_BASE}/prices-history",
                       params={"market": token_ids[0], "interval": "max", "fidelity": 60},
                       timeout=30)
        hist = resp.json().get("history", [])
    except Exception as e:
        print(f"  Error: {question}: {e}")
        continue
    
    if len(hist) < 48:  # skip contracts with < 2 days of data
        continue
    
    df = pd.DataFrame(hist)  # columns are "t" and "p"
    df = df.rename(columns={"t": "timestamp", "p": "price"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"]).set_index("timestamp").sort_index()
    df["condition_id"] = cid
    df["end_date"] = end_date
    df["volume_usd"] = volume
    histories[cid] = df
    print(f"  {question}: {len(df)} pts, vol=${volume:,.0f}")

print(f"\nFetched {len(histories)} contracts with sufficient data")
assert len(histories) > 0, "No contracts fetched — check network / API availability"

In [ ]:
# 1c. Combine into a single DataFrame, resample to daily
all_frames = []
for cid, df in histories.items():
    # Drop rows with NaT index
    df = df[df.index.notna()]
    if df.empty or len(df) < 2:
        continue
    daily = df[["price"]].resample("1D").last().ffill().dropna()
    if daily.empty:
        continue
    daily["condition_id"] = cid
    daily["end_date"] = df["end_date"].iloc[0]
    all_frames.append(daily)

raw_daily = pd.concat(all_frames)
print(f"Combined daily data: {len(raw_daily)} rows, {raw_daily['condition_id'].nunique()} contracts")
print(f"Date range: {raw_daily.index.min().date()} — {raw_daily.index.max().date()}")

## 2. Feature Engineering

In [ ]:
from src.features.builder import FeatureMatrixBuilder, generate_feature_metadata

builder = FeatureMatrixBuilder(
    return_windows=[1, 3, 7, 14],  # daily windows
    sma_windows=[7, 14],
    ema_windows=[7, 14],
    rsi_period=14,
    roc_windows=[1, 3, 7],
)

# Build features per contract, then concatenate
feature_frames = []
for cid, group in raw_daily.groupby("condition_id"):
    end_date = group["end_date"].iloc[0]
    fm = builder.build(group, expiration=end_date)
    feature_frames.append(fm)

features = pd.concat(feature_frames)
print(f"Feature matrix: {features.shape}")
print(f"Contracts: {features['condition_id'].nunique()}")

# Feature metadata
meta = generate_feature_metadata(features)
print(f"\nFeature groups:")
print(meta["group"].value_counts().to_string())

In [ ]:
# Correlation heatmap (Fig 2)
from src.visualization.plots import plot_correlation_heatmap
plot_correlation_heatmap(features, top_n=25)
plt.show()

## 3. Train / Test Split

In [ ]:
TARGET = "target_logret"

# Columns to exclude from features
EXCLUDE_COLS = {
    TARGET, "target_direction", "target_price",
    "condition_id", "end_date",
}

# Prepare X, y — drop rows with NaN in target or features
feature_cols = [c for c in features.select_dtypes(include="number").columns if c not in EXCLUDE_COLS]
subset = features[feature_cols + [TARGET]].dropna()
X = subset[feature_cols]
y = subset[TARGET].values

print(f"Usable rows: {len(X)} (dropped {len(features) - len(X)} with NaNs)")
print(f"Features: {len(feature_cols)}")

# Temporal split — use last 25% as test
split_idx = int(len(X) * 0.75)
X_train_full, y_train_full = X.iloc[:split_idx], y[:split_idx]
X_test, y_test = X.iloc[split_idx:], y[split_idx:]

# Further split training into train + validation (last 15% of train)
val_idx = int(len(X_train_full) * 0.85)
X_train = X_train_full.iloc[:val_idx]
y_train = y_train_full[:val_idx]
X_val = X_train_full.iloc[val_idx:]
y_val = y_train_full[val_idx:]

print(f"\nTrain: {len(X_train)} rows ({X_train.index.min().date()} — {X_train.index.max().date()})")
print(f"Val:   {len(X_val)} rows ({X_val.index.min().date()} — {X_val.index.max().date()})")
print(f"Test:  {len(X_test)} rows ({X_test.index.min().date()} — {X_test.index.max().date()})")

## 4. Model Training

In [ ]:
from src.models.registry import get_model

MODELS_TO_TRAIN = {
    "persistence": {},
    "historical_mean": {"window": 14},
    "ridge": {"alpha": 1.0},
    "lasso": {"alpha": 0.001},
    "lgbm": {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 6},
    "xgboost": {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 6},
}

predictions = {}
trained_models = {}

for name, params in MODELS_TO_TRAIN.items():
    print(f"Training {name}...", end=" ")
    model = get_model(name, **params)
    model.fit(X_train, y_train, X_val, y_val)
    
    preds = model.predict(X_test)
    predictions[name] = preds
    trained_models[name] = model
    
    mask = np.isfinite(preds) & np.isfinite(y_test)
    test_rmse = np.sqrt(np.mean((preds[mask] - y_test[mask]) ** 2)) if mask.sum() > 0 else float("nan")
    print(f"RMSE = {test_rmse:.6f}")

print(f"\nTrained {len(trained_models)} models.")

### 4b. (Optional) LSTM Training

Uncomment to train the LSTM model. Takes a few minutes.

In [ ]:
# # Uncomment to train LSTM
# print("Training LSTM...")
# lstm = get_model("lstm", seq_len=14, hidden_dim=64, n_layers=2,
#                  dropout=0.2, max_epochs=30, batch_size=64, patience=10)
# lstm.fit(X_train_full.values, y_train_full, X_val.values, y_val)
# lstm_preds = lstm.predict(X_test.values)
# 
# mask = np.isfinite(lstm_preds) & np.isfinite(y_test)
# print(f"LSTM RMSE = {np.sqrt(np.mean((lstm_preds[mask] - y_test[mask])**2)):.6f}")
# predictions["lstm"] = lstm_preds
# trained_models["lstm"] = lstm

## 5. Evaluation — Forecast Metrics (RQ1)

In [ ]:
from src.evaluation.metrics import compare_models, hit_rate_by_quintile

metrics_df = compare_models(y_test, predictions)
print("Out-of-sample forecast accuracy:")
display(metrics_df.round(4).style.highlight_min(axis=0, subset=["mae", "rmse", "theils_u"], props="font-weight:bold"))

In [ ]:
# Model comparison bar chart (Fig 3)
from src.visualization.plots import plot_model_comparison
plot_model_comparison(metrics_df, metrics=["mae", "rmse", "directional_accuracy"])
plt.show()

In [ ]:
# Hit rate by quintile of predicted magnitude (best model)
best_model = metrics_df["rmse"].idxmin()
print(f"Hit rate by quintile of predicted magnitude ({best_model}):")
hit_rate_by_quintile(y_test, predictions[best_model])

## 6. Statistical Tests

In [ ]:
from src.evaluation.statistical_tests import (
    dm_pairwise, dm_pvalue_matrix, model_confidence_set, mincer_zarnowitz_all
)

# Diebold-Mariano pairwise
print("=== Diebold-Mariano pairwise tests (H0: equal predictive accuracy) ===")
dm_pairs = dm_pairwise(y_test, predictions, loss="squared")
display(dm_pairs.round(4))

In [ ]:
# DM p-value heatmap (Fig 8)
from src.visualization.plots import plot_dm_heatmap

dm_mat = dm_pvalue_matrix(y_test, predictions)
plot_dm_heatmap(dm_mat)
plt.show()

In [ ]:
# Model Confidence Set
mcs = model_confidence_set(y_test, predictions, alpha=0.10)
print(f"Model Confidence Set (alpha=0.10):")
print(f"  Surviving models: {mcs['surviving_models']}")
print(f"  Eliminated:       {mcs['eliminated']}")
print(f"  Method:           {mcs['method']}")

In [ ]:
# Mincer-Zarnowitz unbiasedness test
mz = mincer_zarnowitz_all(y_test, predictions)
print("Mincer-Zarnowitz regression (H0: alpha=0, beta=1 → unbiased forecasts):")
display(mz.round(4))

## 7. Feature Importance (RQ3)

In [ ]:
from src.visualization.plots import plot_feature_importance

# LightGBM gain-based importance (Fig 5)
if "lgbm" in trained_models:
    lgbm_imp = trained_models["lgbm"].get_feature_importance()
    plot_feature_importance(lgbm_imp, top_n=20, title="LightGBM — Feature Importance (Gain)")
    plt.show()

# Lasso coefficient magnitude (feature selection)
if "lasso" in trained_models:
    lasso_imp = trained_models["lasso"].get_feature_importance()
    nonzero = lasso_imp[lasso_imp > 0]
    print(f"\nLasso selected {len(nonzero)} / {len(lasso_imp)} features")
    if len(nonzero) > 0:
        plot_feature_importance(nonzero, top_n=20, title="Lasso — Non-Zero Coefficients")
        plt.show()

In [ ]:
# Side-by-side importance ranking table (Table 3)
from src.visualization.tables import feature_importance_table

importances = {}
for name in ["lgbm", "xgboost", "ridge", "lasso"]:
    if name in trained_models:
        imp = trained_models[name].get_feature_importance()
        if len(imp) > 0:
            importances[name] = imp

if importances:
    imp_table = feature_importance_table(importances, top_n=15)
    display(imp_table)

## 8. Trading Simulation

In [ ]:
from src.evaluation.trading_sim import compare_strategies, plot_equity_curves

trading_metrics, sims = compare_strategies(
    y_test, predictions,
    strategy="binary",
    threshold=0.0,
    transaction_cost=0.01,  # 1% round-trip
)

print("Trading simulation results (binary strategy, 1% transaction cost):")
display(trading_metrics.round(4))

In [ ]:
# Equity curves (Fig 7)
fig = plot_equity_curves(sims, top_n=5)
plt.show()

## 9. (Optional) Hyperparameter Tuning

Uncomment to run Optuna tuning for selected models. Results are saved to an SQLite DB for reproducibility.

In [ ]:
# # Uncomment to run Optuna tuning
# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
# from src.tuning.optuna_search import tune_model
# 
# # Tune LightGBM (100 trials, ~20 min)
# study_lgbm = tune_model("lgbm", X_train_full.values, y_train_full,
#                          n_trials=100, n_splits=5,
#                          storage="sqlite:///../results/optuna.db")
# print(f"Best LightGBM RMSE: {study_lgbm.best_value:.6f}")
# print(f"Best params: {study_lgbm.best_params}")
# 
# # Retrain with best params
# lgbm_tuned = get_model("lgbm", **study_lgbm.best_params)
# lgbm_tuned.fit(X_train, y_train, X_val, y_val)
# predictions["lgbm_tuned"] = lgbm_tuned.predict(X_test)
# trained_models["lgbm_tuned"] = lgbm_tuned
# 
# # Tune Ridge
# study_ridge = tune_model("ridge", X_train_full.values, y_train_full,
#                           n_trials=50, n_splits=5,
#                           storage="sqlite:///../results/optuna.db")
# print(f"Best Ridge RMSE: {study_ridge.best_value:.6f}, alpha={study_ridge.best_params['alpha']:.4f}")

## 10. Summary

In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\nData: {len(histories)} contracts, {len(X)} usable observations")
print(f"Features: {len(feature_cols)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

print(f"\n--- Best model by RMSE: {best_model} ({metrics_df.loc[best_model, 'rmse']:.6f}) ---")
print(f"--- MCS surviving models: {mcs['surviving_models']} ---")

best_trader = trading_metrics["sharpe_ratio"].idxmax()
print(f"--- Best Sharpe ratio: {best_trader} ({trading_metrics.loc[best_trader, 'sharpe_ratio']:.3f}) ---")

print("\nFull metrics:")
display(metrics_df.round(4))

print("\nTrading performance:")
display(trading_metrics.round(4))